# 🌟 TarjimaAI: English-to-Uzbek Neural Machine Translation (Seq2Seq)
### **Google Colab GPU Accelerated Training & Full Project Google Drive Backup**

This notebook trains both:
1. **Baseline Seq2Seq** (Sutskever et al. 2014 fixed bottleneck)
2. **Bahdanau Additive Attention Seq2Seq** (Bahdanau et al. 2015 dynamic context vectors)

Upon training, it automatically saves **the entire project** (source code, checkpoints, dataset, web app, and documentation) directly to your **Google Drive** both as an uncompressed directory and as a portable `.zip` archive.

## 1. Mount Google Drive
Mount your Google Drive and set up the destination directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Destination folder in your Google Drive where ALL project files will be stored
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/eng_uzb_translator_project'
!mkdir -p "{DRIVE_PROJECT_DIR}"
print(f"Google Drive project folder is ready:\n{DRIVE_PROJECT_DIR}")

## 2. Verify GPU Acceleration (NVIDIA CUDA)

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")

## 3. Clone Repository from GitHub

In [ ]:
import os
REPO_URL = "https://github.com/hamzayevtemur-lab/eng-uzb-translator.git"
REPO_DIR = "/content/eng-uzb-translator"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!ls -la

## 4. Install Dependencies

In [ ]:
%cd {REPO_DIR}
!pip install -r requirements.txt

## 5. Train Both Models on CUDA GPU
- Trains Baseline Seq2Seq model (fixed bottleneck)
- Trains Bahdanau Attention Seq2Seq model (additive dynamic context vectors)
- Generates `vocab_eng.json`, `vocab_uzb.json`, `seq2seq_baseline.pth`, and `seq2seq_attention.pth` in `checkpoints/`

In [ ]:
%cd {REPO_DIR}

# Trains both models sequentially
!python3 src/train.py --model all

## 6. Save ALL Project Files & Checkpoints to Google Drive
Copies the entire project folder (code, trained model weights, dataset, web dashboard, and notebooks) directly into your Google Drive, and additionally packages a compressed `.zip` archive for single-click download.

In [ ]:
%cd {REPO_DIR}

print("1. Copying all project files & checkpoints to Google Drive...")
!cp -rv {REPO_DIR}/* "{DRIVE_PROJECT_DIR}/"

print("\n2. Creating portable .zip archive in Google Drive...")
!zip -r "{DRIVE_PROJECT_DIR}/eng_uzb_translator_complete.zip" {REPO_DIR} -x "*/.git/*"

print("\n" + "=" * 75)
print(f"🎉 ALL PROJECT FILES SUCCESSFULLY SAVED TO GOOGLE DRIVE!")
print(f"Drive Folder: {DRIVE_PROJECT_DIR}")
print("=" * 75)
!ls -lh "{DRIVE_PROJECT_DIR}"

## 7. Interactive Translation & Attention Heatmap Visualizer
Test your freshly trained Bahdanau Attention model on sample sentences directly inside Colab.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")

import torch
from src.dataset import Vocabulary
from src.model_attention import Encoder, Decoder, Seq2SeqAttention
from src.inference import translate_attention, plot_attention_heatmap

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eng_vocab = Vocabulary.load(f"{REPO_DIR}/checkpoints/vocab_eng.json")
uzb_vocab = Vocabulary.load(f"{REPO_DIR}/checkpoints/vocab_uzb.json")

state = torch.load(f"{REPO_DIR}/checkpoints/seq2seq_attention.pth", map_location=device)
enc = Encoder(eng_vocab.n_words, state["embedding_dim"], state["hidden_dim"])
dec = Decoder(uzb_vocab.n_words, state["embedding_dim"], state["hidden_dim"])
model = Seq2SeqAttention(enc, dec, device).to(device)
model.load_state_dict(state["model_state_dict"])
model.eval()

test_sentences = [
    "I love reading books in the library.",
    "Tashkent is the capital city of Uzbekistan.",
    "We drank hot green tea with fresh bread.",
    "Artificial intelligence is transforming modern science and technology."
]

for sent in test_sentences:
    tokens, attn_matrix = translate_attention(sent, model, eng_vocab, uzb_vocab, device=device)
    print("\n" + "=" * 60)
    print(f"Source:      {sent}")
    print(f"Translation: {' '.join(tokens)}")
    plot_attention_heatmap(sent, tokens, attn_matrix)

## 8. Run Architecture Comparison Benchmark

In [ ]:
%cd {REPO_DIR}
!python3 src/compare.py